In [ ]:
RUN_MODE = "observed-dev"  # observed-dev | production
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "OBSERVED_DEV_20260806_01"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
RUN_ID = "E2E_20260806_RESUME_01"
EXECUTE_LIVE = False
MAX_ITEMS = 0
STARTED_AT = "2026-08-06T00:00:00+09:00"

# 03CollectPostingAssets

Thin orchestration notebook for `p4_crawl`.

In [ ]:
from pathlib import Path
import sys

def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "crawl" / "src" / "p4_crawl").is_dir():
            return candidate
        nested = candidate / "DSJA" / "project_4"
        if (nested / "crawl" / "src" / "p4_crawl").is_dir():
            return nested
    raise RuntimeError("Could not locate DSJA/project_4")

PROJECT_ROOT = locate_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "crawl" / "src"))
from p4_crawl.config import RunConfig
from p4_crawl.stage import write_stage_artifacts

config = RunConfig(
    project_root=PROJECT_ROOT,
    run_id=RUN_ID,
    run_mode=RUN_MODE,
    contract_version=CONTRACT_VERSION,
    crawl_release_id=CRAWL_RELEASE_ID,
    data_version=DATA_VERSION,
    as_of_date=AS_OF_DATE,
    random_seed=RANDOM_SEED,
    execute_live=EXECUTE_LIVE,
)

In [ ]:
from p4_crawl.manifests import load_jsonl

asset_manifest = PROJECT_ROOT / "crawl" / "releases" / CRAWL_RELEASE_ID / "asset_manifest.jsonl"
rows = load_jsonl(asset_manifest)
if EXECUTE_LIVE:
    from p4_crawl.cli import main as crawl_cli
    crawl_cli(["--project-root", str(PROJECT_ROOT), "--run-id", RUN_ID, "--run-mode", RUN_MODE,
        "--crawl-release-id", CRAWL_RELEASE_ID, "--execute-live", "assets", "--max-items", str(MAX_ITEMS)])
metrics = {"observedAssetRows": len(rows), "liveExecuted": EXECUTE_LIVE, "status": "ASSET_GAP_DECLARED"}
quality = [
    {"gate": "asset_gap_explicit", "status": "PASS" if len(rows) == 0 else "FAIL", "detail": len(rows)},
    {"gate": "no_live_m1", "status": "PASS" if not EXECUTE_LIVE else "FAIL", "detail": EXECUTE_LIVE},
]
stage_root = write_stage_artifacts(config, "A1-03", STARTED_AT, metrics, quality,
    inputs=[str(asset_manifest.relative_to(PROJECT_ROOT))],
    outputs=["asset_manifest.jsonl", "ocr_candidate_manifest.parquet"])
metrics